# Table S1

Environment: `python_env` (`envs/python_env.yml`)

Input: `out_dir/bulk_deseq2/*.csv` (from `bulk_RNAseq_1_DESeq2.ipynb`)

In [ ]:
from pathlib import Path
import re
import yaml
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

ROOT = Path("..")
config = yaml.safe_load(open(ROOT / "config.yaml"))
OUT_DIR = Path(config["out_dir"])
DE_DIR = OUT_DIR / "bulk_deseq2"
TAB_DIR = OUT_DIR / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

LOG2FC_CUTOFF = 1.5
PADJ_CUTOFF = 0.05

In [ ]:
GROUPS = {0: "Infected vs control",
          1: "Infected: between time points",
          2: "Control: between time points"}

per_tp_re = re.compile(r"^infected_vs_control_(\d+)min\.csv$")
pair_re = re.compile(r"^(infected|control)_(\d+)min_vs_(\d+)min\.csv$")

comparisons = []
for f in DE_DIR.glob("*.csv"):
    m1, m2 = per_tp_re.match(f.name), pair_re.match(f.name)
    if m1:
        tp = m1.group(1)
        key, label = (0, int(tp), 0), f"{tp} min"
    elif m2:
        cond, later, earlier = m2.groups()
        key, label = (1 if cond == "infected" else 2, int(earlier), int(later)), f"{earlier} vs {later} min"
    else:
        continue
    df = pd.read_csv(f, index_col=0)
    comparisons.append({"key": key, "group": GROUPS[key[0]], "label": label, "df": df})
comparisons.sort(key=lambda c: c["key"])
print(f"{len(comparisons)} comparisons loaded")

In [ ]:
def build(direction):
    cols = {}
    for i, c in enumerate(comparisons):
        df = c["df"]
        sig = df["padj"] < PADJ_CUTOFF
        if direction == "up":
            sel = df[sig & (df["log2FoldChange"] > LOG2FC_CUTOFF)]
        else:
            sel = df[sig & (df["log2FoldChange"] < -LOG2FC_CUTOFF)]
        cols[(i, "log2FC")] = sel["log2FoldChange"]
        cols[(i, "padj")] = sel["padj"]
    wide = pd.DataFrame(cols)
    wide = wide.loc[sorted(wide.index, key=lambda g: str(g).lower())]
    print(f"  {direction:4s}: {len(wide)} unique genes")
    return wide


GROUP_FILLS = ["DDEBF7", "FCE4D6", "E2EFDA"]
bold = Font(bold=True)
center = Alignment(horizontal="center", vertical="center", wrap_text=True)
thin = Side(style="thin", color="808080")
thick = Side(style="medium", color="000000")


def write_sheet(ws, wide, direction):
    sign = ">" if direction == "up" else "<"
    thr = LOG2FC_CUTOFF if direction == "up" else -LOG2FC_CUTOFF

    ws.cell(row=1, column=1, value="gene")
    ws.merge_cells(start_row=1, start_column=1, end_row=3, end_column=1)

    first_col = 2
    group_spans = {}
    for i, c in enumerate(comparisons):
        col = first_col + 2 * i
        g = c["key"][0]
        group_spans.setdefault(g, [col, col + 1])[1] = col + 1
        ws.cell(row=2, column=col, value=c["label"])
        ws.merge_cells(start_row=2, start_column=col, end_row=2, end_column=col + 1)
        ws.cell(row=3, column=col, value="log2FC")
        ws.cell(row=3, column=col + 1, value="padj")
    for g, (s, e) in group_spans.items():
        ws.cell(row=1, column=s, value=GROUPS[g])
        ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)

    last_col = first_col + 2 * len(comparisons) - 1
    for r in (1, 2, 3):
        for col in range(1, last_col + 1):
            cell = ws.cell(row=r, column=col)
            cell.font, cell.alignment = bold, center
            if col >= first_col:
                g = comparisons[(col - first_col) // 2]["key"][0]
                cell.fill = PatternFill("solid", fgColor=GROUP_FILLS[g])

    for r, gene in enumerate(wide.index, start=4):
        ws.cell(row=r, column=1, value=gene)
        for i in range(len(comparisons)):
            lfc, padj = wide.at[gene, (i, "log2FC")], wide.at[gene, (i, "padj")]
            if pd.notna(lfc):
                ws.cell(row=r, column=first_col + 2 * i, value=float(lfc)).number_format = "0.00"
                ws.cell(row=r, column=first_col + 2 * i + 1, value=float(padj)).number_format = "0.00E+00"

    last_row = 3 + len(wide)
    group_starts = {s for s, _ in group_spans.values()}
    for col in range(first_col, last_col + 1, 2):
        side = thick if col in group_starts else thin
        for r in range(1, last_row + 1):
            ws.cell(row=r, column=col).border = Border(left=side)

    ws.column_dimensions["A"].width = 14
    for col in range(first_col, last_col + 1):
        ws.column_dimensions[get_column_letter(col)].width = 9.5
    ws.row_dimensions[1].height = 30
    ws.freeze_panes = ws.cell(row=4, column=first_col)

    ws.cell(row=last_row + 2, column=1,
            value=f"Criteria: padj < {PADJ_CUTOFF} and log2FoldChange {sign} {thr}. "
                  f"Blank = not significant in this direction for that comparison.").font = Font(italic=True)

## Table S1

In [ ]:
wb = Workbook()
wb.remove(wb.active)
for direction, sheet in (("up", "Bulk RNAseq upregulated genes"), ("down", "Bulk RNAseq downregulated genes")):
    write_sheet(wb.create_sheet(sheet), build(direction), direction)
wb.save(TAB_DIR / "TableS1.xlsx")